# 03 — Graph-based Molecular Embeddings
**Section A** converts molecules from SDF format to PyTorch Geometric graph objects.
**Section B** loads the trained GNN encoder and extracts 128-dimensional graph embeddings.

> **Note on training:** The GNN encoder was trained in a self-supervised contrastive
> learning framework (SimCLR / NT-Xent loss) on ~5M molecules from ZINC22.
> Pre-trained weights (`gnn_encoder.pt`) are available on Zenodo (see README).

**Output:** CSV with shape `(N_molecules, 128)`.

## Configuration

In [ ]:
INPUT_SDF       = "data/molecules.sdf"          # path to input SDF file
GRAPHS_PT       = "data/graphs.pt"              # where to save/load PyG graphs
MODEL_PT        = "data/gnn_encoder.pt"         # trained GNN weights (from Zenodo)
TRAINING_PT     = "data/training_graphs.pt"     # ZINC22 training graphs (to infer input dim)
OUTPUT_CSV      = "data/graph_embeddings.csv"   # path to output CSV
REFERENCE_SDF     = "data/adagrasib.sdf"        # path to reference molecule SDF file
REFERENCE_CSV_OUT = "data/adagrasib_graph.csv"  # path to reference molecule output CSV file

# GNN hyperparameters (must match training)
HIDDEN_DIM  = 128
NUM_LAYERS  = 3
BATCH_SIZE  = 128
DEVICE = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## Dependencies

In [ ]:
# pip install rdkit torch torch_geometric tqdm pandas numpy

---
## Section A — SDF to Molecular Graphs
Converts each molecule to a PyTorch Geometric `Data` object encoding
atom features (atomic number, degree, valence, charge, hydrogens,
aromaticity, hybridization, chirality, ring membership) and bond features
(bond type, conjugation, ring membership, stereo).

### A.1 Load molecules from SDF

In [ ]:
from rdkit import Chem

supplier = Chem.SDMolSupplier(INPUT_SDF)
mols     = list(supplier)
print(f'Total entries in SDF: {len(mols)}')

### A.2 Define atom and bond feature extraction

In [ ]:
from rdkit.Chem.rdchem import HybridizationType as HT, BondType as BT
import torch
from torch_geometric.data import Data

HYB  = {HT.SP: 0, HT.SP2: 1, HT.SP3: 2, HT.SP3D: 3, HT.SP3D2: 4}
BOND = {BT.SINGLE: 0, BT.DOUBLE: 1, BT.TRIPLE: 2, BT.AROMATIC: 3}

def atom_features(atom):
    return torch.tensor([
        min(atom.GetAtomicNum(), 100),       # atomic number (capped at 100)
        atom.GetTotalDegree(),               # degree
        atom.GetTotalValence(),              # valence
        atom.GetFormalCharge(),              # formal charge
        atom.GetTotalNumHs(),                # hydrogen count
        int(atom.GetIsAromatic()),           # aromaticity
        HYB.get(atom.GetHybridization(), 5), # hybridization
        int(atom.HasProp('_ChiralityPossible')),  # chirality
        int(atom.IsInRing()),                # ring membership
    ], dtype=torch.float)

def bond_features(bond):
    return torch.tensor([
        BOND.get(bond.GetBondType(), 4),     # bond type
        int(bond.GetIsConjugated()),         # conjugation
        int(bond.IsInRing()),                # ring membership
        int(bond.GetStereo()),               # stereo
    ], dtype=torch.float)

def mol_to_pyg(mol):
    x = torch.stack([atom_features(a) for a in mol.GetAtoms()])
    src, dst, eattr = [], [], []
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        bf   = bond_features(b)
        src  += [i, j]; dst += [j, i]; eattr += [bf, bf]
    edge_index = torch.tensor([src, dst], dtype=torch.long)
    edge_attr  = torch.stack(eattr)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

### A.3 Convert molecules to graphs

In [ ]:
from tqdm import tqdm

graphs      = []
bad_indices = []

for i, mol in enumerate(tqdm(mols, desc='Converting')):
    if mol is None:
        bad_indices.append(i)
        continue
    try:
        graphs.append(mol_to_pyg(mol))
    except Exception as e:
        print(f'Error on molecule {i}: {e}')
        bad_indices.append(i)

print(f'Graphs created : {len(graphs)}')
print(f'Bad indices    : {bad_indices}')

### A.4 Save graphs

In [ ]:
torch.save(graphs, GRAPHS_PT)
print(f'Saved: {GRAPHS_PT}')

---
## Section B — Graph Embedding Extraction
Loads the pre-trained GNN encoder and extracts 128-dimensional graph-level
embeddings via global additive pooling over GIN message-passing layers.

### B.1 GNN Encoder architecture
Must match exactly the architecture used during training (see `04_gnn_training.ipynb`).

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch_geometric.nn import GINConv, global_add_pool
from torch_geometric.loader import DataLoader
from torch.utils.data import Dataset


class GNNEncoder(nn.Module):
    """
    GIN-based graph encoder.
    input : x, edge_index, batch
    output: g (graph embedding, dim=HIDDEN_DIM), z (projection head output)
    """
    def __init__(self, in_dim: int, hidden_dim: int = HIDDEN_DIM,
                 num_layers: int = NUM_LAYERS):
        super().__init__()
        self.convs = nn.ModuleList()
        self.bns   = nn.ModuleList()
        dim = in_dim
        for _ in range(num_layers):
            self.convs.append(GINConv(nn.Sequential(
                nn.Linear(dim, hidden_dim), nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim)
            )))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
            dim = hidden_dim
        self.proj_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

    def forward(self, x, edge_index, batch):
        for conv, bn in zip(self.convs, self.bns):
            x = F.relu(bn(conv(x, edge_index)))
        g = global_add_pool(x, batch)
        return g, self.proj_head(g)


class GraphDataset(Dataset):
    def __init__(self, data_list): self.data_list = data_list
    def __len__(self): return len(self.data_list)
    def __getitem__(self, idx): return self.data_list[idx]


print('Architecture defined.')

### B.2 Load pre-trained model

In [ ]:
# Infer input dimension from training graphs
graphs_train = torch.load(TRAINING_PT, weights_only=False)
in_dim = graphs_train[0].x.size(1)
print(f'Input feature dimension: {in_dim}')

model = GNNEncoder(in_dim=in_dim).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PT, weights_only=False, map_location=DEVICE))
model.eval()
print('Model loaded successfully.')

### B.3 Extract embeddings

In [ ]:
graphs_mol  = torch.load(GRAPHS_PT, weights_only=False)
dataset_mol = GraphDataset(graphs_mol)
loader      = DataLoader(dataset_mol, batch_size=BATCH_SIZE, shuffle=False)

embeddings = []
with torch.no_grad():
    for batch in tqdm(loader, desc='Extracting embeddings'):
        batch = batch.to(DEVICE)
        g, _  = model(batch.x, batch.edge_index, batch.batch)
        embeddings.append(g.cpu().numpy())

emb = np.concatenate(embeddings, axis=0)
print(f'Embedding matrix shape: {emb.shape}')  # (N, 128)
print(f'Negative values: {np.mean(emb < 0)*100:.1f}%')

### B.4 Save to CSV

In [ ]:
df = pd.DataFrame(emb)
df.to_csv(OUTPUT_CSV, index=False)
print(f'Saved: {OUTPUT_CSV}  |  shape: {df.shape}')

---
## Section C — Reference Molecule Graph Embedding

### C.1 Convert reference molecule to graph

In [ ]:
from rdkit import Chem

supplier_ref = Chem.SDMolSupplier(REFERENCE_SDF)
mol_ref = [mol for mol in supplier_ref if mol is not None]
assert len(mol_ref) == 1

graph_ref     = mol_to_pyg(mol_ref[0])
dataset_ref   = GraphDataset([graph_ref])
loader_ref    = DataLoader(dataset_ref, batch_size=1, shuffle=False)

### C.2 Extract reference embedding

In [ ]:
with torch.no_grad():
    for batch in loader_ref:
        batch   = batch.to(DEVICE)
        g_ref, _ = model(batch.x, batch.edge_index, batch.batch)
        ref_emb  = g_ref.cpu().numpy()

df_ref = pd.DataFrame(ref_emb)
df_ref.to_csv(REFERENCE_CSV_OUT, index=False)
print(f"Saved: {REFERENCE_CSV_OUT}  |  shape: {df_ref.shape}")